# Year-to-Year Temporal Validation: 2020-2025

This notebook performs year-to-year temporal validation to test model generalization across consecutive seasons:

- **2020 → 2021**: Train on 2019-20 only, test on 2020-21
- **2021 → 2022**: Train on 2020-21 only, test on 2021-22
- **2022 → 2023**: Train on 2021-22 only, test on 2022-23
- **2023 → 2024**: Train on 2022-23 only, test on 2023-24
- **2024 → 2025**: Train on 2023-24 only, test on 2024-25

For each fold, we report:
- ROC AUC, Log Loss, Brier Score, Accuracy (exact numbers)
- Training set size and test set size
- Model: Gradient Boosting (best performer from ablation tests)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score
import matplotlib.pyplot as plt

print("Imports complete")

## 2. Load All Seasons

In [ ]:
# Define season mappings
seasons = {
    2020: 'enriched_data/nbastatsv3_2019_enriched_shots.csv',  # 2019-20 season
    2021: 'enriched_data/nbastatsv3_2020_enriched_shots.csv',  # 2020-21 season
    2022: 'enriched_data/nbastatsv3_2021_enriched_shots.csv',  # 2021-22 season
    2023: 'enriched_data/nbastatsv3_2022_enriched_shots.csv',  # 2022-23 season
    2024: 'enriched_data/nbastatsv3_2023_enriched_shots.csv',  # 2023-24 season
    2025: 'enriched_data/nbastatsv3_2024_enriched_shots.csv',  # 2024-25 season
}

# Load all seasons
data = {}
for year, path in seasons.items():
    try:
        df = pd.read_csv(path)
        # Create binary target
        df['target'] = (df['shotResult'].astype(str).str.strip().str.lower() == 'made').astype(int)
        data[year] = df
        print(f"Loaded {year}: {len(df):,} shots, FG% = {df['target'].mean():.1%}")
    except FileNotFoundError:
        print(f"⚠️  {year}: File not found at {path}")

print(f"\nLoaded {len(data)} seasons")

## 3. Define Features & Preprocessing

In [ ]:
# Feature definitions (excluding leakage columns)
LEAKAGE_COLUMNS = ['actionType', 'description', 'shotResult', 'isFieldGoal']

# Numeric features
NUMERIC_FEATURES = [
    'shotDistance',
    'SHOT_CLOCK_APPROX',
    'xLegacy',
    'yLegacy',
    'period',
    'contest_score',
    'shotValue',
    'ABS_TIME'
]

# Categorical features (excluding leakage)
CATEGORICAL_FEATURES = [
    'subType',
    'contest_label',
    'teamTricode',
    'location'
]

# Preprocessing pipeline (dense for tree-based models)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='drop'
)

print(f"Numeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")

## 4. Year-to-Year Temporal Validation

In [ ]:
# Define year-to-year validation folds (train on single year, test on next year)
folds = [
    {'train_year': 2020, 'test_year': 2021, 'name': '2020 → 2021'},
    {'train_year': 2021, 'test_year': 2022, 'name': '2021 → 2022'},
    {'train_year': 2022, 'test_year': 2023, 'name': '2022 → 2023'},
    {'train_year': 2023, 'test_year': 2024, 'name': '2023 → 2024'},
    {'train_year': 2024, 'test_year': 2025, 'name': '2024 → 2025'},
]

results = []

print("="*80)
print("YEAR-TO-YEAR TEMPORAL VALIDATION")
print("="*80)

for fold in folds:
    train_year = fold['train_year']
    test_year = fold['test_year']
    fold_name = fold['name']
    
    print(f"\n{'='*80}")
    print(f"FOLD: {fold_name}")
    print(f"{'='*80}")
    print(f"Training on: {train_year} only")
    print(f"Testing on: {test_year}")
    
    # Check if required seasons are available
    if train_year not in data:
        print(f"⚠️  Skipping: Training year {train_year} not available")
        continue
    if test_year not in data:
        print(f"⚠️  Skipping: Test year {test_year} not available")
        continue
    
    # Get single year training data and test data
    train_df = data[train_year]
    test_df = data[test_year]
    
    # Extract features and target
    all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
    
    X_train = train_df[all_features]
    y_train = train_df['target']
    X_test = test_df[all_features]
    y_test = test_df['target']
    
    print(f"\nTraining set: {len(X_train):,} shots ({y_train.mean():.1%} made)")
    print(f"Test set: {len(X_test):,} shots ({y_test.mean():.1%} made)")
    
    # Create and train model
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            random_state=42,
            verbose=0
        ))
    ])
    
    print("\nTraining model...")
    model.fit(X_train, y_train)
    
    # Predict on test set
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # Calculate metrics
    test_auc = roc_auc_score(y_test, y_pred_proba)
    test_logloss = log_loss(y_test, y_pred_proba, labels=[0, 1])
    test_brier = brier_score_loss(y_test, y_pred_proba)
    test_acc = accuracy_score(y_test, y_pred)
    
    print(f"\n{'─'*80}")
    print("TEST SET RESULTS:")
    print(f"{'─'*80}")
    print(f"  ROC AUC:      {test_auc:.4f}")
    print(f"  Log Loss:     {test_logloss:.4f}")
    print(f"  Brier Score:  {test_brier:.4f}")
    print(f"  Accuracy:     {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"{'─'*80}")
    
    # Store results
    results.append({
        'fold': fold_name,
        'train_year': train_year,
        'test_year': test_year,
        'train_size': len(X_train),
        'test_size': len(X_test),
        'train_fg_pct': y_train.mean(),
        'test_fg_pct': y_test.mean(),
        'test_auc': test_auc,
        'test_logloss': test_logloss,
        'test_brier': test_brier,
        'test_accuracy': test_acc
    })

print("\n" + "="*80)
print("ALL FOLDS COMPLETE")
print("="*80)

## 5. Summary Results Table

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)

# Display full table
print("\n" + "="*80)
print("YEAR-TO-YEAR TEMPORAL VALIDATION SUMMARY")
print("="*80)
print()

# Format for display
display_df = results_df[[
    'fold',
    'train_size',
    'test_size',
    'test_auc',
    'test_logloss',
    'test_brier',
    'test_accuracy'
]].copy()

display_df.columns = ['Fold', 'Train N', 'Test N', 'AUC', 'Log Loss', 'Brier', 'Accuracy']

print(display_df.to_string(index=False))
print()

# Calculate average metrics
print("\n" + "─"*80)
print("AVERAGE METRICS ACROSS ALL FOLDS:")
print("─"*80)
print(f"  Mean AUC:      {results_df['test_auc'].mean():.4f} ± {results_df['test_auc'].std():.4f}")
print(f"  Mean Log Loss: {results_df['test_logloss'].mean():.4f} ± {results_df['test_logloss'].std():.4f}")
print(f"  Mean Brier:    {results_df['test_brier'].mean():.4f} ± {results_df['test_brier'].std():.4f}")
print(f"  Mean Accuracy: {results_df['test_accuracy'].mean():.4f} ± {results_df['test_accuracy'].std():.4f}")
print(f"                 ({results_df['test_accuracy'].mean()*100:.2f}% ± {results_df['test_accuracy'].std()*100:.2f}%)")
print("─"*80)

## 6. Visualization: Performance Over Time

In [ ]:
# Plot metrics over time
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Year-to-Year Temporal Validation: Performance Across Consecutive Seasons', fontsize=16, fontweight='bold')

metrics = [
    ('test_auc', 'ROC AUC', axes[0, 0], (0.85, 1.0)),
    ('test_logloss', 'Log Loss', axes[0, 1], None),
    ('test_brier', 'Brier Score', axes[1, 0], None),
    ('test_accuracy', 'Accuracy', axes[1, 1], (0.85, 1.0))
]

for metric_col, metric_name, ax, ylim in metrics:
    ax.plot(range(len(results_df)), results_df[metric_col], 
            marker='o', markersize=10, linewidth=2.5, color='steelblue')
    ax.set_xlabel('Year Pair', fontsize=11)
    ax.set_ylabel(metric_name, fontsize=11)
    ax.set_title(metric_name, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels(results_df['fold'], rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3)
    if ylim:
        ax.set_ylim(ylim)
    
    # Add value labels
    for i, val in enumerate(results_df[metric_col]):
        ax.text(i, val, f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Export Results

In [ ]:
# Export detailed results
output_file = 'year_to_year_temporal_validation_results.csv'
results_df.to_csv(output_file, index=False)
print(f"✅ Results saved to: {output_file}")

# Print summary statistics
print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)
print("""
Key Findings:

1. **Year-to-Year Generalization**: Each model trained on only one season
   - Tests if shot quality patterns are stable year-over-year
   - If AUC stays ~0.90+, features generalize well to next season
   - If metrics vary significantly, suggests NBA evolution or rule changes

2. **Consistency Check**: All folds use equal training data (1 season)
   - Compare performance across different year pairs
   - Similar metrics = stable shot quality patterns
   - Varying metrics = changing NBA landscape

3. **Expected Performance**:
   - AUC: 0.88-0.95 (shot quality is highly predictable)
   - Accuracy: 85-92% (consistent with published research)
   - Log Loss: 0.20-0.35 (well-calibrated probabilities)

4. **What to Look For**:
   - Stable metrics across years = robust features
   - Declining metrics = NBA evolution (e.g., rule changes, 3PT revolution)
   - Specific year drops = identify what changed that season
""")
print("="*80)

## Notes

### Data Requirements
This notebook requires enriched shot data for seasons 2019-20 through 2024-25:
- `enriched_data/nbastatsv3_2019_enriched_shots.csv`
- `enriched_data/nbastatsv3_2020_enriched_shots.csv`
- `enriched_data/nbastatsv3_2021_enriched_shots.csv`
- `enriched_data/nbastatsv3_2022_enriched_shots.csv`
- `enriched_data/nbastatsv3_2023_enriched_shots.csv`
- `enriched_data/nbastatsv3_2024_enriched_shots.csv`

### Model Configuration
- **Algorithm**: Gradient Boosting Classifier
- **Hyperparameters**: n_estimators=100, learning_rate=0.1, max_depth=5
- **Features**: Same as main modeling notebook (excluding actionType to prevent leakage)

### Validation Strategy
- **Year-to-year**: Train on single year, test on next year
- **No cumulative training**: Each fold uses only one training year
- **No overlap**: Test data never appears in training

### Expected Runtime
- ~15-20 minutes total (depends on data size)
- Each fold: ~2-4 minutes